# Part 1: Tiny Transformer (Problems 001–009)

In this notebook we build the core of a GPT-style language model from scratch:

1. **Vocabulary** — map tokens ↔ integer IDs
2. **Encoding/Decoding** — convert text to IDs and back
3. **Causal Mask** — prevent attending to future tokens
4. **Scaled Dot-Product Attention** — the heart of the transformer
5. **Multi-Head Attention + FFN** — full transformer block
6. **GPT Forward Pass** — end-to-end on a toy example
7. **Load DistilGPT2** — run real generation with pretrained weights

## Cell 1: Build vocabulary from a sample text corpus

In [ ]:
try:
    from solutions.p001_build_token_vocab import build_token_vocab
except (ImportError, NotImplementedError, ModuleNotFoundError):
    try:
        import importlib
        _m = importlib.import_module("solutions.001_build_token_vocab")
        build_token_vocab = _m.build_token_vocab
    except Exception:
        print("Solve problem 001 first:")
        print("  cp problems/001_build_token_vocab.py solutions/001_build_token_vocab.py")
        build_token_vocab = None

corpus = """
the quick brown fox jumps over the lazy dog
the dog barked at the fox
the fox ran away quickly
a quick brown dog jumped over a lazy cat
""".strip()

if build_token_vocab is not None:
    token_to_id, id_to_token = build_token_vocab(corpus)
    print(f"Vocabulary size: {len(token_to_id)} unique tokens")
    print()
    print("Token → ID mapping:")
    for token, idx in list(token_to_id.items())[:12]:
        print(f"  '{token}' → {idx}")
    if len(token_to_id) > 12:
        print(f"  ... and {len(token_to_id) - 12} more")

## Cell 2: Encode a sentence, decode it back

In [ ]:
try:
    import importlib
    _m = importlib.import_module("solutions.002_encode_and_decode")
    encode = _m.encode
    decode = _m.decode
except Exception:
    print("Solve problem 002 first:")
    print("  cp problems/002_encode_and_decode.py solutions/002_encode_and_decode.py")
    encode = decode = None

if encode is not None and build_token_vocab is not None:
    token_to_id, id_to_token = build_token_vocab(corpus)

    sentence = "the quick brown fox"
    ids = encode(sentence, token_to_id)
    recovered = decode(ids, id_to_token)

    print(f"Original sentence : '{sentence}'")
    print(f"Encoded (token IDs): {ids}")
    print(f"Decoded back       : '{recovered}'")
    print()
    assert recovered == sentence, f"Round-trip failed: got '{recovered}'"
    print("Round-trip check passed!")

## Cell 3: Visualise the causal mask as a heatmap

In [ ]:
try:
    import importlib
    _m = importlib.import_module("solutions.003_build_causal_mask")
    build_causal_mask = _m.build_causal_mask
except Exception:
    print("Solve problem 003 first:")
    print("  cp problems/003_build_causal_mask.py solutions/003_build_causal_mask.py")
    build_causal_mask = None

import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt
import numpy as np

if build_causal_mask is not None:
    seq_len = 8
    mask = build_causal_mask(seq_len)

    # Convert to numpy for plotting
    if hasattr(mask, 'numpy'):
        mask_np = mask.numpy()
    else:
        mask_np = np.array(mask)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(mask_np, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"Causal Mask (seq_len={seq_len})\nBlue = attend, White = masked")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig("/tmp/causal_mask.png", dpi=100)
    plt.show()
    print(f"Mask shape: {mask_np.shape}")
    print("Lower-triangular = each token attends to itself and all previous tokens")
else:
    # Fallback: show what it should look like
    seq_len = 8
    mask_np = np.tril(np.ones((seq_len, seq_len)))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(mask_np, cmap="Blues")
    ax.set_title(f"Expected Causal Mask shape (seq_len={seq_len})\n[implement problem 003 to generate this]")
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.tight_layout()
    plt.show()

## Cell 4: Scaled dot-product attention on random tensors

In [ ]:
try:
    import importlib
    _m = importlib.import_module("solutions.004_scaled_dot_product_attention")
    scaled_dot_product_attention = _m.scaled_dot_product_attention
except Exception:
    print("Solve problem 004 first:")
    print("  cp problems/004_scaled_dot_product_attention.py solutions/004_scaled_dot_product_attention.py")
    scaled_dot_product_attention = None

import torch

torch.manual_seed(42)

batch_size = 2
seq_len = 6
d_k = 32  # key/query dimension
d_v = 32  # value dimension

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_v)

print(f"Input shapes:")
print(f"  Q: {Q.shape}")
print(f"  K: {K.shape}")
print(f"  V: {V.shape}")

if scaled_dot_product_attention is not None:
    output = scaled_dot_product_attention(Q, K, V)
    print(f"\nOutput shape: {output.shape}")
    print(f"Output[0, 0, :6] = {output[0, 0, :6].tolist()}")
    print()
    print("Formula: softmax(QK^T / sqrt(d_k)) * V")
    print(f"Scaling factor: 1/sqrt({d_k}) = {1/(d_k**0.5):.4f}")

## Cell 5: Full transformer forward pass end-to-end

In [ ]:
try:
    import importlib
    _m = importlib.import_module("solutions.008_gpt_model_forward")
    gpt_model_forward = _m.gpt_model_forward
except Exception:
    print("Solve problem 008 first:")
    print("  cp problems/008_gpt_model_forward.py solutions/008_gpt_model_forward.py")
    gpt_model_forward = None

import torch

torch.manual_seed(0)

# Toy GPT config
config = {
    "vocab_size": 50,
    "n_positions": 16,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 2,
    "d_ff": 128,
}

# Fake input: batch of 2 sequences, length 5
token_ids = torch.randint(0, config["vocab_size"], (2, 5))

print("Toy GPT config:")
for k, v in config.items():
    print(f"  {k}: {v}")
print(f"\nInput token_ids shape: {token_ids.shape}")
print(f"Input token_ids:\n{token_ids}")

if gpt_model_forward is not None:
    logits = gpt_model_forward(token_ids, config)
    print(f"\nOutput logits shape: {logits.shape}")
    print(f"  [batch=2, seq_len=5, vocab_size={config['vocab_size']}]")
    print(f"\nNext-token logits for first sequence, last position:")
    print(logits[0, -1, :10].tolist())

## Cell 6: Load DistilGPT2 weights and generate a sentence

In [ ]:
try:
    import importlib
    _m = importlib.import_module("solutions.009_load_pretrained_weights")
    load_pretrained_weights = _m.load_pretrained_weights
except Exception:
    print("Solve problem 009 first:")
    print("  cp problems/009_load_pretrained_weights.py solutions/009_load_pretrained_weights.py")
    load_pretrained_weights = None

try:
    import importlib
    _m = importlib.import_module("solutions.016_generate_with_prompt")
    generate_with_prompt = _m.generate_with_prompt
except Exception:
    print("Solve problem 016 first:")
    print("  cp problems/016_generate_with_prompt.py solutions/016_generate_with_prompt.py")
    generate_with_prompt = None

if load_pretrained_weights is not None:
    print("Loading DistilGPT2 weights (requires HuggingFace transformers)...")
    model, tokenizer = load_pretrained_weights("distilgpt2")
    print(f"Model loaded: {type(model).__name__}")

    if generate_with_prompt is not None:
        prompt = "The future of artificial intelligence"
        print(f"\nPrompt: '{prompt}'")
        print("Generating...")
        generated = generate_with_prompt(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=20,
            temperature=0.8,
        )
        print(f"Generated: '{generated}'")